# Face Detection Demo with MediaPipe

This notebook demonstrates face detection and landmark extraction using MediaPipe Face Landmarker. We'll explore:

1. Loading and displaying video
2. Detecting faces and extracting 478 3D facial landmarks
3. Visualizing different landmark groups (eyes, eyebrows, mouth, face oval)
4. Interactive visualization of landmark connections

## Setup

First, let's import the necessary libraries and set up our environment.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import cv2
import matplotlib.pyplot as plt
from matplotlib import patches
import warnings
warnings.filterwarnings('ignore')

# Add parent directory to path to import asdrp
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from asdrp import MediaPipeFaceDetector, VideoFileReader
from asdrp.visualization import (
    FaceOverlay, OverlayStyle,
    LEFT_EYE_CONNECTIONS, RIGHT_EYE_CONNECTIONS,
    LEFT_EYEBROW_CONNECTIONS, RIGHT_EYEBROW_CONNECTIONS,
    MOUTH_CONNECTIONS, NOSE_CONNECTIONS,
    FACE_OVAL_CONNECTIONS, ALL_CONNECTIONS
)

# Configure matplotlib for better display
plt.rcParams['figure.figsize'] = (15, 10)
plt.rcParams['figure.dpi'] = 100

print("Setup complete!")

## 1. Video Loading and Display

Let's load the downloaded YouTube video and display some basic information about it.

In [ ]:
# Path to video file
video_path = project_root / "data" / "videos" / "youtube_short_emotion.mp4"

# Check if video exists
if not video_path.exists():
    print(f"Error: Video not found at {video_path}")
    print("Please ensure the video file is downloaded to data/videos/")
else:
    print(f"Video found: {video_path}")
    print(f"File size: {video_path.stat().st_size / (1024*1024):.2f} MB")

# Open video and get metadata
with VideoFileReader(str(video_path)) as reader:
    metadata = reader.metadata
    print(f"\nVideo Metadata:")
    print(f"  Resolution: {metadata.width}x{metadata.height}")
    print(f"  FPS: {metadata.fps:.2f}")
    print(f"  Total frames: {metadata.total_frames}")
    print(f"  Duration: {metadata.duration:.2f} seconds")
    print(f"  Codec: {metadata.codec}")

Let's display a few sample frames from the video.

In [ ]:
# Display sample frames
with VideoFileReader(str(video_path)) as reader:
    # Get frames at different timestamps
    frame_indices = [0, metadata.total_frames // 4, metadata.total_frames // 2, 
                     3 * metadata.total_frames // 4, metadata.total_frames - 1]
    
    fig, axes = plt.subplots(1, 5, figsize=(20, 4))
    
    for idx, frame_num in enumerate(frame_indices):
        frame_data = reader.get_frame_at(frame_num)
        if frame_data:
            # Convert BGR to RGB for display
            rgb_frame = cv2.cvtColor(frame_data.frame, cv2.COLOR_BGR2RGB)
            axes[idx].imshow(rgb_frame)
            axes[idx].set_title(f"Frame {frame_num}\n({frame_data.timestamp_ms/1000:.2f}s)")
            axes[idx].axis('off')
    
    plt.tight_layout()
    plt.show()

## 2. Face Detection and Landmark Extraction

Now let's initialize the MediaPipe Face Detector and detect faces in the video. MediaPipe can detect up to 478 3D facial landmarks per face.

**Note:** You'll need to download the MediaPipe Face Landmarker model from:
https://developers.google.com/mediapipe/solutions/vision/face_landmarker

Place it in the `models/` directory as `face_landmarker.task`

In [ ]:
# Path to the MediaPipe model
model_path = project_root / "models" / "face_landmarker.task"

# Check if model exists
if not model_path.exists():
    print(f"\n⚠️  Model not found at {model_path}")
    print("\nPlease download the MediaPipe Face Landmarker model:")
    print("1. Visit: https://developers.google.com/mediapipe/solutions/vision/face_landmarker")
    print("2. Download face_landmarker_v2_with_blendshapes.task")
    print(f"3. Place it in: {model_path.parent}/")
    print("4. Rename to: face_landmarker.task")
    
    # Create models directory if it doesn't exist
    model_path.parent.mkdir(exist_ok=True)
    print(f"\nCreated directory: {model_path.parent}/")
else:
    print(f"✓ Model found: {model_path}")
    print(f"  File size: {model_path.stat().st_size / (1024*1024):.2f} MB")

In [ ]:
# Initialize face detector (only run if model exists)
if model_path.exists():
    detector = MediaPipeFaceDetector(
        model_path=str(model_path),
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5,
        num_faces=1,
        running_mode="VIDEO"
    )
    print("Face detector initialized successfully!")
    print(f"Detector configuration: {detector}")
else:
    print("Skipping detector initialization (model not found)")

Let's detect faces in a sample frame and display the landmarks.

In [ ]:
# Detect faces in a sample frame
if model_path.exists():
    with VideoFileReader(str(video_path)) as reader:
        # Get a frame from the middle of the video
        frame_num = metadata.total_frames // 2
        frame_data = reader.get_frame_at(frame_num)
        
        if frame_data:
            # Detect faces
            faces = detector.detect(frame_data.frame, timestamp_ms=frame_data.timestamp_ms)
            
            print(f"\nDetection results for frame {frame_num}:")
            print(f"  Number of faces detected: {len(faces)}")
            
            if faces:
                face = faces[0]
                print(f"\nFace 0 information:")
                print(f"  Total landmarks: {face.num_landmarks}")
                print(f"  Landmark shape: {face.landmarks.shape}")
                print(f"  Bounding box: ({face.bounding_box.x_min:.3f}, {face.bounding_box.y_min:.3f}, "
                      f"{face.bounding_box.width:.3f}, {face.bounding_box.height:.3f})")
                print(f"  Has visibility: {face.visibility is not None}")
                
                # Display first few landmarks
                print(f"\nSample landmarks (first 5):")
                for i in range(5):
                    lm = face.landmarks[i]
                    print(f"    Landmark {i}: x={lm[0]:.4f}, y={lm[1]:.4f}, z={lm[2]:.4f}")
            else:
                print("  No faces detected in this frame")

## 3. Visualizing Facial Landmarks

Let's visualize the detected landmarks on the face. We'll start with all landmarks, then show specific facial regions.

In [ ]:
# Helper function to draw landmarks
def draw_landmarks_on_frame(frame, face_landmarks, connections=None, 
                           landmark_color=(0, 255, 0), 
                           connection_color=(255, 255, 255),
                           landmark_radius=2,
                           connection_thickness=1):
    """Draw landmarks and connections on a frame."""
    annotated_frame = frame.copy()
    h, w = frame.shape[:2]
    
    landmarks = face_landmarks.landmarks
    
    # Draw connections first (so they appear behind landmarks)
    if connections:
        for start_idx, end_idx in connections:
            start_point = landmarks[start_idx]
            end_point = landmarks[end_idx]
            
            start_px = (int(start_point[0] * w), int(start_point[1] * h))
            end_px = (int(end_point[0] * w), int(end_point[1] * h))
            
            cv2.line(annotated_frame, start_px, end_px, connection_color, connection_thickness)
    
    # Draw landmarks
    for landmark in landmarks:
        x_px = int(landmark[0] * w)
        y_px = int(landmark[1] * h)
        cv2.circle(annotated_frame, (x_px, y_px), landmark_radius, landmark_color, -1)
    
    return annotated_frame

print("Helper function defined!")

### 3.1 All Landmarks Visualization

In [ ]:
# Display frame with all landmarks
if model_path.exists() and faces:
    face = faces[0]
    
    # Create visualization with all landmarks
    annotated = draw_landmarks_on_frame(
        frame_data.frame,
        face,
        connections=ALL_CONNECTIONS,
        landmark_color=(0, 255, 0),
        connection_color=(255, 255, 255),
        landmark_radius=1,
        connection_thickness=1
    )
    
    # Display
    plt.figure(figsize=(12, 10))
    plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    plt.title(f"All 478 Facial Landmarks\nFrame {frame_num} @ {frame_data.timestamp_ms/1000:.2f}s", 
              fontsize=14, fontweight='bold')
    plt.axis('off')
    plt.tight_layout()
    plt.show()

### 3.2 Landmark Groups by Facial Region

Let's visualize different facial regions separately to better understand the landmark structure.

In [ ]:
# Visualize different facial regions
if model_path.exists() and faces:
    face = faces[0]
    
    # Define regions and their connections
    regions = [
        ("Left Eye", LEFT_EYE_CONNECTIONS, (255, 100, 0)),
        ("Right Eye", RIGHT_EYE_CONNECTIONS, (0, 100, 255)),
        ("Left Eyebrow", LEFT_EYEBROW_CONNECTIONS, (255, 200, 0)),
        ("Right Eyebrow", RIGHT_EYEBROW_CONNECTIONS, (0, 200, 255)),
        ("Mouth", MOUTH_CONNECTIONS, (255, 0, 100)),
        ("Nose", NOSE_CONNECTIONS, (100, 255, 0)),
        ("Face Oval", FACE_OVAL_CONNECTIONS, (200, 200, 200)),
    ]
    
    # Create subplots
    fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    axes = axes.flatten()
    
    # Original frame
    axes[0].imshow(cv2.cvtColor(frame_data.frame, cv2.COLOR_BGR2RGB))
    axes[0].set_title("Original Frame", fontsize=12, fontweight='bold')
    axes[0].axis('off')
    
    # Each region
    for idx, (region_name, connections, color) in enumerate(regions):
        annotated = draw_landmarks_on_frame(
            frame_data.frame,
            face,
            connections=connections,
            landmark_color=color,
            connection_color=color,
            landmark_radius=3,
            connection_thickness=2
        )
        
        axes[idx + 1].imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
        axes[idx + 1].set_title(region_name, fontsize=12, fontweight='bold')
        axes[idx + 1].axis('off')
    
    plt.tight_layout()
    plt.suptitle("Facial Landmark Groups by Region", fontsize=16, fontweight='bold', y=1.02)
    plt.show()

### 3.3 Combined Regions Visualization

In [ ]:
# Draw all regions with different colors on one frame
if model_path.exists() and faces:
    face = faces[0]
    annotated = frame_data.frame.copy()
    h, w = annotated.shape[:2]
    
    # Define all regions with distinct colors
    region_configs = [
        (LEFT_EYE_CONNECTIONS, (255, 100, 0), "Left Eye"),
        (RIGHT_EYE_CONNECTIONS, (0, 100, 255), "Right Eye"),
        (LEFT_EYEBROW_CONNECTIONS, (255, 200, 0), "Left Eyebrow"),
        (RIGHT_EYEBROW_CONNECTIONS, (0, 200, 255), "Right Eyebrow"),
        (MOUTH_CONNECTIONS, (255, 0, 100), "Mouth"),
        (NOSE_CONNECTIONS, (100, 255, 0), "Nose"),
        (FACE_OVAL_CONNECTIONS, (200, 200, 200), "Face Oval"),
    ]
    
    landmarks = face.landmarks
    
    # Draw each region
    for connections, color, name in region_configs:
        for start_idx, end_idx in connections:
            start_point = landmarks[start_idx]
            end_point = landmarks[end_idx]
            
            start_px = (int(start_point[0] * w), int(start_point[1] * h))
            end_px = (int(end_point[0] * w), int(end_point[1] * h))
            
            cv2.line(annotated, start_px, end_px, color, 2)
        
        # Draw landmarks for this region
        unique_indices = set()
        for start_idx, end_idx in connections:
            unique_indices.add(start_idx)
            unique_indices.add(end_idx)
        
        for idx in unique_indices:
            landmark = landmarks[idx]
            x_px = int(landmark[0] * w)
            y_px = int(landmark[1] * h)
            cv2.circle(annotated, (x_px, y_px), 2, color, -1)
    
    # Display
    plt.figure(figsize=(14, 12))
    plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    plt.title("All Facial Regions with Color-Coded Landmarks", fontsize=16, fontweight='bold')
    plt.axis('off')
    
    # Add legend
    from matplotlib.patches import Patch
    legend_elements = [Patch(facecolor=np.array(color[::-1])/255, label=name) 
                      for _, color, name in region_configs]
    plt.legend(handles=legend_elements, loc='upper left', bbox_to_anchor=(1.02, 1), fontsize=11)
    
    plt.tight_layout()
    plt.show()

## 4. Landmark Detection Across Multiple Frames

Let's see how landmarks are tracked across different frames in the video.

In [ ]:
# Process multiple frames
if model_path.exists():
    with VideoFileReader(str(video_path)) as reader:
        # Sample frames throughout the video
        num_samples = 6
        frame_indices = np.linspace(0, metadata.total_frames - 1, num_samples, dtype=int)
        
        results = []
        for frame_num in frame_indices:
            frame_data = reader.get_frame_at(frame_num)
            if frame_data:
                faces = detector.detect(frame_data.frame, timestamp_ms=frame_data.timestamp_ms)
                results.append((frame_data, faces))
        
        # Visualize
        fig, axes = plt.subplots(2, 3, figsize=(18, 12))
        axes = axes.flatten()
        
        for idx, (frame_data, faces) in enumerate(results):
            if faces:
                annotated = draw_landmarks_on_frame(
                    frame_data.frame,
                    faces[0],
                    connections=ALL_CONNECTIONS,
                    landmark_color=(0, 255, 0),
                    connection_color=(255, 255, 255),
                    landmark_radius=1,
                    connection_thickness=1
                )
                axes[idx].imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
                axes[idx].set_title(f"Frame {frame_data.frame_number}\n"
                                   f"Time: {frame_data.timestamp_ms/1000:.2f}s\n"
                                   f"Faces: {len(faces)}",
                                   fontsize=10)
            else:
                axes[idx].imshow(cv2.cvtColor(frame_data.frame, cv2.COLOR_BGR2RGB))
                axes[idx].set_title(f"Frame {frame_data.frame_number}\nNo faces detected", 
                                   fontsize=10)
            axes[idx].axis('off')
        
        plt.suptitle("Face Detection Across Video Timeline", fontsize=16, fontweight='bold')
        plt.tight_layout()
        plt.show()

## 5. Landmark Position Analysis

Let's analyze the 3D positions of landmarks and their distribution.

In [ ]:
# Analyze landmark positions
if model_path.exists() and faces:
    face = faces[0]
    landmarks = face.landmarks
    
    # Extract x, y, z coordinates
    x_coords = landmarks[:, 0]
    y_coords = landmarks[:, 1]
    z_coords = landmarks[:, 2]
    
    # Create visualizations
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    # 2D scatter (front view)
    axes[0, 0].scatter(x_coords, y_coords, c=z_coords, cmap='viridis', s=10, alpha=0.6)
    axes[0, 0].set_xlabel('X (horizontal)', fontsize=11)
    axes[0, 0].set_ylabel('Y (vertical)', fontsize=11)
    axes[0, 0].set_title('2D Landmark Distribution (Front View)\nColor = Depth (Z)', 
                        fontsize=12, fontweight='bold')
    axes[0, 0].invert_yaxis()  # Invert Y to match image coordinates
    axes[0, 0].set_aspect('equal')
    
    # X-Z scatter (side view)
    axes[0, 1].scatter(z_coords, x_coords, c=y_coords, cmap='plasma', s=10, alpha=0.6)
    axes[0, 1].set_xlabel('Z (depth)', fontsize=11)
    axes[0, 1].set_ylabel('X (horizontal)', fontsize=11)
    axes[0, 1].set_title('Landmark Distribution (Side View)\nColor = Y position', 
                        fontsize=12, fontweight='bold')
    
    # Depth histogram
    axes[1, 0].hist(z_coords, bins=50, color='skyblue', edgecolor='black', alpha=0.7)
    axes[1, 0].set_xlabel('Z coordinate (depth)', fontsize=11)
    axes[1, 0].set_ylabel('Count', fontsize=11)
    axes[1, 0].set_title('Depth Distribution of Landmarks', fontsize=12, fontweight='bold')
    axes[1, 0].axvline(z_coords.mean(), color='red', linestyle='--', 
                       label=f'Mean: {z_coords.mean():.4f}')
    axes[1, 0].legend()
    
    # Statistics table
    stats_text = f"""
    Landmark Statistics:
    
    X (Horizontal):
      Range: [{x_coords.min():.4f}, {x_coords.max():.4f}]
      Mean: {x_coords.mean():.4f}
      Std: {x_coords.std():.4f}
    
    Y (Vertical):
      Range: [{y_coords.min():.4f}, {y_coords.max():.4f}]
      Mean: {y_coords.mean():.4f}
      Std: {y_coords.std():.4f}
    
    Z (Depth):
      Range: [{z_coords.min():.4f}, {z_coords.max():.4f}]
      Mean: {z_coords.mean():.4f}
      Std: {z_coords.std():.4f}
    
    Total landmarks: {len(landmarks)}
    """
    axes[1, 1].text(0.1, 0.5, stats_text, fontsize=11, family='monospace',
                   verticalalignment='center')
    axes[1, 1].axis('off')
    axes[1, 1].set_title('Landmark Statistics', fontsize=12, fontweight='bold')
    
    plt.tight_layout()
    plt.show()

## 6. Interactive Landmark Explorer

Let's create an interactive view to explore specific landmarks.

In [ ]:
# Highlight specific landmark indices
if model_path.exists() and faces:
    face = faces[0]
    
    # Define some key landmark indices
    key_landmarks = {
        "Left Eye Center": [33, 133],
        "Right Eye Center": [362, 263],
        "Nose Tip": [1],
        "Mouth Corners": [61, 291],
        "Chin": [152],
        "Forehead": [10],
    }
    
    # Create visualization
    annotated = frame_data.frame.copy()
    h, w = annotated.shape[:2]
    landmarks = face.landmarks
    
    # Draw all landmarks faintly
    for landmark in landmarks:
        x_px = int(landmark[0] * w)
        y_px = int(landmark[1] * h)
        cv2.circle(annotated, (x_px, y_px), 1, (200, 200, 200), -1)
    
    # Draw key landmarks with labels
    colors = [(255, 0, 0), (0, 255, 0), (0, 0, 255), (255, 255, 0), (255, 0, 255), (0, 255, 255)]
    
    for (name, indices), color in zip(key_landmarks.items(), colors):
        for idx in indices:
            landmark = landmarks[idx]
            x_px = int(landmark[0] * w)
            y_px = int(landmark[1] * h)
            
            # Draw larger circle for key landmarks
            cv2.circle(annotated, (x_px, y_px), 5, color, -1)
            cv2.circle(annotated, (x_px, y_px), 7, (255, 255, 255), 2)
            
            # Add label
            cv2.putText(annotated, f"{name} ({idx})", (x_px + 10, y_px - 10),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 1, cv2.LINE_AA)
    
    # Display
    plt.figure(figsize=(14, 12))
    plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    plt.title("Key Facial Landmarks Highlighted", fontsize=16, fontweight='bold')
    plt.axis('off')
    plt.tight_layout()
    plt.show()
    
    # Print landmark coordinates
    print("\nKey Landmark Coordinates:")
    print("-" * 70)
    for name, indices in key_landmarks.items():
        print(f"\n{name}:")
        for idx in indices:
            lm = landmarks[idx]
            print(f"  Index {idx:3d}: x={lm[0]:.4f}, y={lm[1]:.4f}, z={lm[2]:.4f}")

## Summary

In this notebook, we've covered:

1. Loading and displaying video files
2. Initializing the MediaPipe Face Detector
3. Detecting faces and extracting 478 3D facial landmarks
4. Visualizing all landmarks and connections
5. Exploring different facial regions (eyes, eyebrows, mouth, nose, face oval)
6. Analyzing landmark positions in 3D space
7. Tracking landmarks across multiple frames
8. Highlighting and exploring key facial landmarks

### Next Steps

- **02_emotion_analysis_demo.ipynb**: Learn how to use these landmarks for emotion detection
- **03_temporal_analysis.ipynb**: Analyze emotion changes over time

### Clean Up

In [ ]:
# Clean up resources
if model_path.exists():
    detector.close()
    print("Detector closed successfully!")